# Modifications

Compared to try 2:

Firstly, we see significant improvements from try 1 to try 2 in regards to F1 score. However, the results are still quite bad. Improvement suggestion: Adjust batch size and learning rate.

In [ ]:
# Step 1: Load the data for training
import sys
sys.path.append("./functions/")

from functions.helpers import load_csv_data
from functions.implementations import reg_logistic_regression_weighted
from functions.implementations import lr_inference
import matplotlib.pyplot as plt
import numpy as np
import json
import random

random.seed(12102026)
np.random.seed(12102026)

_, _, y_train, _, _ = load_csv_data("./dataset/", sub_sample=False) # Switch off sub_sample the moment you aren't debugging.
x_train_new = np.loadtxt("./processed_dataset/X_processed.csv", delimiter=",")

# Fix y_train for sigmoid
y_train = (y_train == 1).reshape(len(y_train), 1) # 1 when 1, 0 otherwise.

# Add bias term at train time.
bias = np.ones(len(x_train_new)).reshape(len(x_train_new), 1)
x_train_new = np.concatenate((x_train_new, bias), axis=1)

# 80% Training, 20% Validation
split_location = int(0.80*len(x_train_new))
x_t = x_train_new[0:split_location]
y_t = y_train[0:split_location]
x_v = x_train_new[split_location:]
y_v = y_train[split_location:]

In [ ]:
# Step 2: Configure Hyperparameters (and other properties)
MAX_ITER = 1 # Max number of iterations
BATCH_SIZE = 1 # Size of 1 batch
gamma = 2e-4 # Learning Rate
lambda_ = 0.00002 # Lambda (Regularization)

losses_train = [] # To be calculated once after each iteration
losses_validation = [] # To be calculated once after each iteration
accuracy_validation = [] # The accuracy on the validation set
f1_validation = [] # F1 Score on the validation set
network_size = 254 # The number of weights in the layer of the model, currently equal to the number of features
weights = np.random.normal(loc=0, scale=np.sqrt(2/(x_t.shape[1])), size=network_size) # Initial Weights of the model. Based on Kaiming initialization.
weights = weights.reshape(len(weights), 1)
dataset_bias = len(y_t[y_t == 1])/len(y_t)
dataset_bias = dataset_bias*5.0 # Try to modify it, to see whether different changes makes it better or worse.

In [ ]:
# Step 3: Train the model
for i in range(MAX_ITER):
    t_loss = []
    v_loss = []
    v_acc = [] # Validation Accuracy
    v_f1 = [] # Validation F1 Score
    v_tp_c = 0 # True Positive
    v_fp_c = 0 # False Positive
    v_fn_c = 0 # False Negative
    # Training
    for j in range(0, len(x_t), BATCH_SIZE):
        x_t_batch = x_t[BATCH_SIZE*j:min(len(x_t), BATCH_SIZE*(j+1))]
        y_t_batch = y_t[BATCH_SIZE*j:min(len(y_t), BATCH_SIZE*(j+1))]
        losses, weights = reg_logistic_regression_weighted(y_t_batch, x_t_batch, lambda_, weights, gamma, dataset_bias)
        t_loss.extend(losses)

    # Validation Loss
    for j in range(len(x_v)):
        losses, _ = reg_logistic_regression_weighted(y_v[j:(j+1)], x_v[j:(j+1)], lambda_, weights, gamma, dataset_bias) # Do not update the weights
        classification = lr_inference(x_v[j:(j+1)], weights)
        v_loss.extend(losses)
        v_acc.append(classification == y_v[j:(j+1)])
        if classification[0, 0] == 1 and y_v[j:(j+1), 0] == 1:
            v_tp_c += 1
        elif classification[0, 0] == 1 and y_v[j:(j+1), 0] == 0:
            v_fp_c += 1
        elif classification[0, 0] == 0 and y_v[j:(j+1), 0] == 1:
            v_fn_c += 1

    # This assumes that each calculated loss value has the same weight inherently.
    losses_train.append(sum(t_loss)/len(t_loss))
    losses_validation.append(sum(v_loss)/len(v_loss))
    accuracy_validation.append((sum(v_acc)/len(v_acc))[0, 0])
    f1_validation.append((2*v_tp_c)/(2*v_tp_c + v_fp_c + v_fn_c))

    print(f"Training Loss: {sum(t_loss)/len(t_loss)} ; Validation Loss: {sum(v_loss)/len(v_loss)} ; Val Acc: {(sum(v_acc)/len(v_acc))[0, 0]} ; Val F1: {(2*v_tp_c)/(2*v_tp_c + v_fp_c + v_fn_c)}")

    if i % 5 == 0:
        print(f"{i+1}/{MAX_ITER} completed.")

In [ ]:
# Step 4: Graph training results
fig, ax = plt.subplots(2, 1)

ax[0].plot(np.arange(len(losses_train)), losses_train)
ax[0].set(xlabel="Iteration", ylabel="Loss", title="Training Loss")

ax[1].plot(np.arange(len(losses_validation)), losses_validation)
ax[1].set(xlabel="Iteration", ylabel="Loss", title="Validation Loss")

fig.show()

In [ ]:
# Step 5: Save the data
model_name = "simple_model_try3"
np.savetxt(f"./models/{model_name}.txt", weights, delimiter=",")
with open(f"./models/{model_name}_train_loss.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(losses_train))

with open(f"./models/{model_name}_validation_loss.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(losses_validation))

with open(f"./models/{model_name}_validation_acc.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(accuracy_validation))

with open(f"./models/{model_name}_validation_f1.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(f1_validation))

# Conclusions from modifications

Initially, the model was too regularized, so reducing regularization gave good results.

Furthermore, the bias towards the positive class was too high, so reducing the bias towards it was advantageous. Seemingly scaling by 5 was optimal.

Increasing the learning rate was also seemingly good for this problem.

Iterations dropped to just 1, since iterating longer was bad on the validation set, apparently.